In [0]:
# Spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ML utilities
import numpy as np
import pandas as pd

# XGBoost
from xgboost import XGBClassifier

# Sklearn helpers
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("F1-Strategy-Engine")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

In [0]:
MASTER_PATH = "workspace.default.f1_cleaned_lap_dataset"

df = spark.table(MASTER_PATH)

In [0]:
display(df)

In [0]:
stint_window = (
    Window
    .partitionBy("Driver", "Year", "Circuit", "Stint")
    .orderBy("LapNumber")
)

df = df.withColumn(
    "lap_delta",
    F.col("LapTime") - F.lag("LapTime").over(stint_window)
)

df = df.withColumn(
    "lap_delta",
    F.when((F.col("lap_delta") < -5) | (F.col("lap_delta") > 5), None)
     .otherwise(F.col("lap_delta"))
)

In [0]:
driver_style = (
    df
    .groupBy("Driver", "Year")
    .agg(
        F.avg("lap_delta").alias("avg_deg_rate"),
        F.stddev("lap_delta").alias("deg_variance"),
        F.avg("TyreLife").alias("avg_stint_length"),
        F.avg("HasPit").alias("pit_frequency")
    )
    .fillna(0)
)

In [0]:
for c in ["avg_deg_rate", "deg_variance", "avg_stint_length", "pit_frequency"]:
    stats = driver_style.select(
        F.mean(c).alias("mean"),
        F.stddev(c).alias("std")
    ).collect()[0]

    driver_style = driver_style.withColumn(
        c, (F.col(c) - stats["mean"]) / (stats["std"] + 1e-6)
    )

In [0]:
df = df.join(driver_style, on=["Driver", "Year"], how="left")

# Team profile mirrors driver profiling but captures how a constructor performs over time
team_season_profile = (
    df
    .groupBy("TeamName", "Year")
    .agg(
        F.avg("QualiPosition").alias("team_avg_quali"),
        F.avg("Position").alias("team_avg_race_pos"),
        F.avg("DeltaToLeader").alias("team_avg_delta_to_leader"),
        F.avg("HasPit").alias("team_pit_aggression"),
        F.stddev("Position").alias("team_position_variability")
    )
    .fillna(0)
)

# Historical-only view: cumulative profile from previous seasons for each team
team_hist_window = (
    Window
    .partitionBy("TeamName")
    .orderBy("Year")
    .rowsBetween(Window.unboundedPreceding, -1)
)

team_profile = (
    team_season_profile
    .withColumn("team_hist_avg_quali", F.avg("team_avg_quali").over(team_hist_window))
    .withColumn("team_hist_avg_race_pos", F.avg("team_avg_race_pos").over(team_hist_window))
    .withColumn("team_hist_car_pace", F.avg("team_avg_delta_to_leader").over(team_hist_window))
    .withColumn("team_hist_strategy_aggr", F.avg("team_pit_aggression").over(team_hist_window))
    .withColumn("team_hist_consistency", F.avg("team_position_variability").over(team_hist_window))
    .fillna(0)
)

for c in [
    "team_hist_avg_quali",
    "team_hist_avg_race_pos",
    "team_hist_car_pace",
    "team_hist_strategy_aggr",
    "team_hist_consistency"
]:
    stats = team_profile.select(F.mean(c).alias("mean"), F.stddev(c).alias("std")).collect()[0]
    team_profile = team_profile.withColumn(
        c, (F.col(c) - stats["mean"]) / (stats["std"] + 1e-6)
    )

df = df.join(
    team_profile.select(
        "TeamName", "Year",
        "team_hist_avg_quali",
        "team_hist_avg_race_pos",
        "team_hist_car_pace",
        "team_hist_strategy_aggr",
        "team_hist_consistency"
    ),
    on=["TeamName", "Year"],
    how="left"
)


### **Step 2: Target Construction**

In [0]:
finish_window = Window.partitionBy("Year", "Circuit", "Driver")

df = df.withColumn(
    "FinalPosition",
    F.max("Position").over(finish_window)
)

In [0]:
df = (
    df
    .withColumn("PodiumFinish", (F.col("FinalPosition") <= 3).cast("int"))
    .withColumn("WinFinish", (F.col("FinalPosition") == 1).cast("int"))
)

In [0]:
weather_window = Window.partitionBy("Year", "Circuit")

df = (
    df
    .withColumn("AirTemp_delta", F.col("AirTemp_C") - F.avg("AirTemp_C").over(weather_window))
    .withColumn("TrackTemp_delta", F.col("TrackTemp_C") - F.avg("TrackTemp_C").over(weather_window))
    .withColumn("Humidity_delta", F.col("Humidity_pct") - F.avg("Humidity_pct").over(weather_window))
    .withColumn("WindSpeed_delta", F.col("WindSpeed_kmh") - F.avg("WindSpeed_kmh").over(weather_window))
    .drop("AirTemp_C", "TrackTemp_C", "Humidity_pct", "WindSpeed_kmh")
)

### **Step 3: Feature Selection**

In [0]:
FEATURES_PRERACE = [
    "Driver",
    "TeamName",
    "Circuit",
    "Year",
    "QualiPosition",

    # Driver style
    "avg_deg_rate",
    "deg_variance",
    "avg_stint_length",
    "pit_frequency",

    # Team historical profile
    "team_hist_avg_quali",
    "team_hist_avg_race_pos",
    "team_hist_car_pace",
    "team_hist_strategy_aggr",
    "team_hist_consistency"
]


In [0]:
FEATURES_LIVE = [
    "LapNumber",
    "RacePhase",
    "Position",
    "GapToAhead",
    "DeltaToLeader",

    "Compound",
    "TyreLife",
    "Stint",
    "TrackStatus",

    # Driver style (allowed)
    "avg_deg_rate",
    "deg_variance",
    "avg_stint_length",
    "pit_frequency",

    # Team historical profile
    "team_hist_avg_quali",
    "team_hist_avg_race_pos",
    "team_hist_car_pace",
    "team_hist_strategy_aggr",
    "team_hist_consistency",

    # Weather (relative only)
    "AirTemp_delta",
    "TrackTemp_delta",
    "Humidity_delta",
    "WindSpeed_delta"
]


### **Step 4: Convert to Pandas**

In [0]:
df_prerace = df.select(FEATURES_PRERACE + ["PodiumFinish"])
df_live = df.select(FEATURES_LIVE + ["PodiumFinish"])

pdf_prerace = df_prerace.toPandas()
pdf_live = df_live.toPandas()

In [0]:
CATEGORICAL_PRERACE = ["Driver", "TeamName", "Circuit"]
CATEGORICAL_LIVE = ["Compound", "RacePhase"]

for c in CATEGORICAL_PRERACE:
    pdf_prerace[c] = pdf_prerace[c].astype("category")

for c in CATEGORICAL_LIVE:
    pdf_live[c] = pdf_live[c].astype("category")

### **Step 6: Train Pre-Race Strategy Probability Model**

In [0]:
X_pre = pdf_prerace[FEATURES_PRERACE]
y_pre = pdf_prerace["PodiumFinish"]

groups = (
    pdf_prerace["Year"].astype(str) +
    "_" +
    pdf_prerace["Circuit"].astype(str)
)

gss = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_pre, y_pre, groups=groups))

X_pre_train = X_pre.iloc[train_idx]
y_pre_train = y_pre.iloc[train_idx]

X_pre_test = X_pre.iloc[test_idx]
y_pre_test = y_pre.iloc[test_idx]

In [0]:
pre_model = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist"
)

pre_model.fit(X_pre_train, y_pre_train)


In [0]:
from sklearn.metrics import roc_auc_score

probs = pre_model.predict_proba(X_pre_test)[:, 1]
auc = roc_auc_score(y_pre_test, probs)

print("Pre-race AUC (unseen races):", round(auc, 4))


### **Step 5: Train Live Podium Probability Model**

In [0]:
groups = pdf_live.index  # lap-wise independence

gss = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(pdf_live, groups=groups))

X_train = pdf_live.iloc[train_idx][FEATURES_LIVE]
y_train = pdf_live.iloc[train_idx]["PodiumFinish"]

X_test = pdf_live.iloc[test_idx][FEATURES_LIVE]
y_test = pdf_live.iloc[test_idx]["PodiumFinish"]

live_model = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist"
)

live_model.fit(X_train, y_train)


In [0]:
probs = live_model.predict_proba(X_test)[:, 1]
print("LIVE MODEL AUC:", roc_auc_score(y_test, probs))

In [0]:
baseline_podium = pre_model.predict_proba(X_pre.iloc[[0]])[0][1]
live_podium = live_model.predict_proba(X_test.iloc[[0]])[0][1]

delta = live_podium - baseline_podium
print(delta)

In [0]:
import matplotlib.pyplot as plt
from xgboost import plot_importance

plot_importance(pre_model, max_num_features=15)
plt.show()

In [0]:
probs = pre_model.predict_proba(X_pre)[:, 1]
roc_auc_score(y_pre, probs)

In [0]:
from sklearn.calibration import calibration_curve

# predicted probabilities
probs = pre_model.predict_proba(X_pre)[:, 1]

# calibration data
prob_true, prob_pred = calibration_curve(
    y_pre,
    probs,
    n_bins=10,
    strategy="uniform"
)

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))

# Perfect calibration reference
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")

# Your model
plt.plot(prob_pred, prob_true, marker="o", label="Pre-race model")

plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Calibration curve (Pre-race model)")
plt.legend()
plt.grid(True)

plt.show()

In [0]:
live_probs = live_model.predict_proba(X_test)[:, 1]

prob_true_l, prob_pred_l = calibration_curve(
    y_test,
    live_probs,
    n_bins=10
)

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], "k--")
plt.plot(prob_pred_l, prob_true_l, marker="o")
plt.title("Calibration curve (Live model)")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.grid(True)
plt.show()


In [0]:
# Save Model to local filesystem only (serverless clusters cannot write to /dbfs/FileStore)
pre_model.save_model("/tmp/pre_model.json")
live_model.save_model("/tmp/live_model.json")

# Note: Copying to /dbfs/FileStore/models/ is not supported on serverless clusters. Models are saved in /tmp and can be downloaded from there if needed.